In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "cryptohftdata"])
subprocess.run([sys.executable, "-m", "pip", "install", "polars"])

## Download all Files (Multi-Core)

In [ ]:
import pandas as pd
import cryptohftdata as chd
from datetime import datetime, timedelta
import os
import gc
from concurrent.futures import ProcessPoolExecutor, as_completed

# --- CONFIGURATION ---
API_KEY = "467ab24e39287d1280d30cf4b3cd803217ac2b73bac1e71201251117c7cad952"
START_DATE = "2025-08-01"
END_DATE = "2025-08-30"
SYMBOL = "BTCUSDT"
EXCHANGE = chd.exchanges.BINANCE_FUTURES
MAX_WORKERS = 6  # <- tune this (start low!)

DROP_HEADERS = [
    'some_unused_column', 'symbol',
    'transaction_time', 'received_time', 'timestamp'
]


def optimize_floats(df):
    floats = df.select_dtypes(include=['float64']).columns
    df[floats] = df[floats].astype('float32')
    return df


def get_date_list(start, end):
    start_dt = datetime.strptime(start, "%Y-%m-%d")
    end_dt = datetime.strptime(end, "%Y-%m-%d")
    delta = end_dt - start_dt
    return [(start_dt + timedelta(days=i)).strftime("%Y-%m-%d")
            for i in range(delta.days + 1)]


def process_single_date(date_str, category):
    """Runs inside each process."""
    client = chd.CryptoHFTDataClient(api_key=API_KEY)

    cat_name, fetch_method_name = category
    fetch_func = getattr(client, fetch_method_name)

    os.makedirs(cat_name, exist_ok=True)
    file_path = f"orderbook_raw/{date_str}_{SYMBOL}.parquet"

    if os.path.exists(file_path):
        return f"[SKIP] {date_str}"

    try:
        df = fetch_func(
            symbol=SYMBOL,
            exchange=EXCHANGE,
            start_date=date_str,
            end_date=date_str
        )

        if df is None or df.empty:
            return f"[EMPTY] {date_str}"

        # Drop unused columns
        df.drop(columns=[c for c in DROP_HEADERS if c in df.columns],
                errors='ignore', inplace=True)

        # Convert time
        if 'received_time' in df.columns:
            df['received_time'] = pd.to_datetime(df['received_time'], unit='ns')

        # Optimize
        df = optimize_floats(df)

        # Save
        df.to_parquet(
            file_path,
            engine='pyarrow',
            compression='zstd',
            compression_level=9,
            index=False
        )

        rows = len(df)

        del df
        gc.collect()

        return f"[SAVED] {date_str} | Rows: {rows:,}"

    except Exception as e:
        if 'df' in locals():
            del df
        gc.collect()
        return f"[ERROR] {date_str}: {e}"


def download_and_store():
    dates = get_date_list(START_DATE, END_DATE)

    categories = [
        ("orderbook", "get_orderbook"),
        #("trades", "get_trades"),
        #("open_interest", "get_open_interest"),
    ]

    for category in categories:
        cat_name, _ = category
        print(f"\n>>> Starting Category: {cat_name.upper()}")

        with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = [
                executor.submit(process_single_date, date, category)
                for date in dates
            ]

            for future in as_completed(futures):
                print(" ", future.result())


if __name__ == "__main__":
    download_and_store()
    print("\nDownload process complete.")

## Now Split files into multiple segments:

In [ ]:
import polars as pl
from tqdm import tqdm
import os

def split_parquet_by_interval(input_file, num_intervals, prefix):
    # 1. Scan the file lazily
    lf = pl.scan_parquet(input_file)

    # 2. Get the start and end times
    # Since it's already datetime[ns], we just pull the min/max
    print("Analyzing file metadata...")
    time_bounds = lf.select([
        pl.col("received_time").min().alias("start"),
        pl.col("received_time").max().alias("end")
    ]).collect()

    start_time = time_bounds["start"][0]
    end_time = time_bounds["end"][0]

    if start_time is None or end_time is None:
        print("Error: Could not find time bounds. Is the 'received_time' column empty?")
        return

    total_duration = end_time - start_time
    interval_duration = total_duration / num_intervals

    print(f"Start: {start_time}")
    print(f"End:   {end_time}")
    print(f"Splitting into {num_intervals} chunks of ~{interval_duration} each.\n")

    # 3. Create output directory
    output_dir = "orderbook_chunks"
    os.makedirs(output_dir, exist_ok=True)

    # 4. Filter and save in a memory-efficient loop
    for i in tqdm(range(num_intervals), desc="Processing Chunks"):
        chunk_start = start_time + (i * interval_duration)
        chunk_end = start_time + ((i + 1) * interval_duration)

        # We use collect() inside the loop so only ONE slice is in RAM at a time
        chunk_df = (
            lf.filter(
                (pl.col("received_time") >= chunk_start) & 
                (pl.col("received_time") < chunk_end)
            )
            .collect()
        )

        if not chunk_df.is_empty():
            output_filename = f"{output_dir}/{prefix}_chunk_{i+1:02d}.parquet"
            chunk_df.write_parquet(output_filename)
        
        # Explicitly clear chunk from memory (optional, but good for very tight RAM)
        del chunk_df

    print(f"\nDone! Files are in the '{output_dir}' folder.")

import os

# Assuming split_parquet_by_interval is defined above...

if __name__ == "__main__":
    # Specify the folder containing your parquet files
    folder_path = "./orderbook_raw" 
    
    if not os.path.exists(folder_path):
        print(f"Error: Folder '{folder_path}' not found.")
    else:
        # Get a list of all parquet files in the directory
        files = [f for f in os.listdir(folder_path) if f.endswith('.parquet')]
        
        if not files:
            print("No .parquet files found in the directory.")
        else:
            try:
                intervals = int(input("Enter the number of intervals (e.g., 24 for hourly): "))
                
                for filename in files:
                    # Construct the full path to the file
                    file_path = os.path.join(folder_path, filename)
                    
                    print(f"--- Processing: {filename} ---")
                    split_parquet_by_interval(file_path, intervals, filename)
                
                print("\nAll files processed successfully.")
                
            except ValueError:
                print("Please enter a valid whole number.")

## Now reconstuct it?

## Reconstruction by Gemini:

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from decimal import Decimal

# Configuration
INPUT_FOLDER = "orderbook_chunks"
OUTPUT_FOLDER = "features"
OUTPUT_FILE = "orderbook_features.parquet"
INTERVAL_NS = 30 * 1_000_000_000  # 30 seconds in nanoseconds
LEVELS_TO_EXTRACT = 10  # Number of levels to use for feature computation

def compute_features(bids, asks, timestamp):
    """
    Computes ML features from the current state of the order book.
    """
    # Sort bids descending, asks ascending
    sorted_bids = sorted(bids.items(), key=lambda x: x[0], reverse=True)
    sorted_asks = sorted(asks.items(), key=lambda x: x[0])
    
    # Extract top N levels (convert to float for ML computation speed)
    top_bids = [(float(p), float(q)) for p, q in sorted_bids[:LEVELS_TO_EXTRACT]]
    top_asks = [(float(p), float(q)) for p, q in sorted_asks[:LEVELS_TO_EXTRACT]]
    
    # Initialize feature dictionary
    features = {'timestamp': timestamp}
    
    # If the book is empty on either side, we can't compute meaningful features
    if not top_bids or not top_asks:
        return None
        
    best_bid, best_bid_qty = top_bids[0]
    best_ask, best_ask_qty = top_asks[0]
    
    # 1. Standard L1 Features
    features['best_bid'] = best_bid
    features['best_ask'] = best_ask
    features['mid_price'] = (best_bid + best_ask) / 2.0
    features['spread'] = best_ask - best_bid
    features['spread_bps'] = (features['spread'] / features['mid_price']) * 10000
    
    # 2. Microprice (Volume-weighted mid price)
    # High microprice relative to mid price indicates strong buying pressure
    features['micro_price'] = (best_bid * best_ask_qty + best_ask * best_bid_qty) / (best_bid_qty + best_ask_qty)
    
    # 3. Depth & Volume Features (Aggregated over 1, 5, and N levels)
    for n in [1, 5, LEVELS_TO_EXTRACT]:
        bid_vol = sum(q for p, q in top_bids[:n])
        ask_vol = sum(q for p, q in top_asks[:n])
        
        features[f'bid_vol_{n}'] = bid_vol
        features[f'ask_vol_{n}'] = ask_vol
        features[f'total_vol_{n}'] = bid_vol + ask_vol
        
        # 4. Order Book Imbalance (OBI)
        # Ranges from -1 (heavy sell pressure) to 1 (heavy buy pressure)
        if (bid_vol + ask_vol) > 0:
            features[f'obi_{n}'] = (bid_vol - ask_vol) / (bid_vol + ask_vol)
        else:
            features[f'obi_{n}'] = 0.0

    # 5. Price Level Densities (Average size per price step)
    bid_price_range = top_bids[0][0] - top_bids[-1][0]
    ask_price_range = top_asks[-1][0] - top_asks[0][0]
    
    features['bid_density'] = features[f'bid_vol_{LEVELS_TO_EXTRACT}'] / bid_price_range if bid_price_range > 0 else 0
    features['ask_density'] = features[f'ask_vol_{LEVELS_TO_EXTRACT}'] / ask_price_range if ask_price_range > 0 else 0

    return features

def process_orderbook_stream():
    # State Persistence across files
    bids = {}  # Format: {Decimal(price): Decimal(quantity)}
    asks = {}
    
    # Find and sort all parquet files to ensure chronological order
    file_pattern = os.path.join(INPUT_FOLDER, "*.parquet")
    files = sorted(glob.glob(file_pattern))
    
    if not files:
        print("No parquet files found in the specified directory.")
        return

    features_list = []
    next_snapshot_time = None
    
    print(f"Found {len(files)} files. Beginning processing...")

    for file in files:
        print(f"Processing: {file}")
        # Read the file
        df = pd.read_parquet(file)
        
        # Iterating through the dataframe rows
        for row in df.itertuples(index=False):
            # Row attributes based on your schema
            ts = row.received_time
            side = row.side
            price = Decimal(row.price)
            quantity = Decimal(row.quantity)
            
            # Initialize snapshot timer on very first row
            if next_snapshot_time is None:
                # Align to a clean 30-second boundary (optional but recommended)
                next_snapshot_time = ts + INTERVAL_NS
            
            # --- INTERVAL CHECK ---
            # If current row's timestamp crosses our 30-second threshold
            while ts >= next_snapshot_time:
                # Take snapshot and compute features
                feats = compute_features(bids, asks, next_snapshot_time)
                if feats:
                    features_list.append(feats)
                
                # Advance the snapshot timer
                next_snapshot_time += INTERVAL_NS
            
            # --- LOB UPDATE ---
            # Update the order book with the current row
            target_book = bids if side == 'bid' else asks
            
            if quantity == Decimal('0'):
                # Quantity 0 means delete the price level
                target_book.pop(price, None)
            else:
                # Update or insert new quantity at price level
                target_book[price] = quantity

    # Create DataFrame from collected features
    features_df = pd.DataFrame(features_list)
    
    if features_df.empty:
        print("No features generated. Stream might be empty.")
        return

    # --- TARGET VARIABLE CREATION ---
    # We want to predict if the price is higher or lower in 15 minutes.
    # 15 minutes = 900 seconds. Since our intervals are 30 seconds, 
    # a 15-minute look-ahead is exactly 30 rows ahead.
    
    PERIODS_15_MIN = int((15 * 60 * 1_000_000_000) / INTERVAL_NS) # 30
    
    # Shift the mid_price backwards by 30 rows to get the "future" price
    features_df['future_mid_price_15m'] = features_df['mid_price'].shift(-PERIODS_15_MIN)
    
    # Create the binary classification target: 1 if future price > current price, else 0
    # Note: The last 30 rows will have NaN for future_mid_price_15m. We can drop them.
    features_df['target_price_up'] = (features_df['future_mid_price_15m'] > features_df['mid_price']).astype(int)
    
    # Drop rows where we don't have a future price (the end of the dataset)
    features_df = features_df.dropna(subset=['future_mid_price_15m'])

    # Save to Parquet
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    output_path = os.path.join(OUTPUT_FOLDER, OUTPUT_FILE)
    features_df.to_parquet(output_path, engine='pyarrow', index=False)
    
    print(f"Successfully processed files. Saved features to {output_path}")

if __name__ == "__main__":
    process_orderbook_stream()